[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aai-services/datainsight-curriculum/blob/main/stages/stage-1/1.4-combining-and-reshaping/lesson.ipynb)

# 1.4 Combining and reshaping

Week 6.

By the end of this lesson you can:

- Summarise by group with `groupby`
- Join two tables with `merge`, and check the join did what you intended
- Reshape tables between wide and long formats

We use separate country tables for life expectancy, population, and region (Our World in Data, CC BY 4.0).

In [ ]:
import pandas as pd

DATA = "https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/datasets/"

life = pd.read_csv(DATA + "life_expectancy.csv")
pop = pd.read_csv(DATA + "population.csv")
regions = pd.read_csv(DATA + "regions.csv")

life.head()

## Summaries by group

`groupby` splits the table into groups, applies a calculation to each, and combines the results.

In [ ]:
life.groupby("year")["life_expectancy"].median().tail()

`.agg` computes several summaries at once:

In [ ]:
life.groupby("year")["life_expectancy"].agg(["count", "median", "min", "max"]).tail()

## Joining tables

`merge` matches rows from two tables on shared columns. The most important habit: **count rows before and after every join**. A join that silently adds or loses rows is one of the most common errors in data analysis.

In [ ]:
print("life rows:", len(life))
print("regions rows:", len(regions))

joined = life.merge(regions[["code", "region"]], on="code", how="left", validate="many_to_one")
print("joined rows:", len(joined))
assert len(joined) == len(life)

`how="left"` keeps every row of `life`, even where no region matches. `validate="many_to_one"` makes pandas stop with an error if `regions` unexpectedly has two rows for the same country, which would duplicate rows in the result.

Here every country found a region. That is not always so. Join GDP per person onto life expectancy, and use `indicator=True` to see which rows found a match:

In [ ]:
gdp = pd.read_csv(DATA + "gdp_per_capita.csv")
check = life.merge(gdp, on=["country", "code", "year"], how="left", indicator=True, validate="one_to_one")
check["_merge"].value_counts()

In [ ]:
check.loc[check["_merge"] == "left_only", "country"].drop_duplicates().head(12).tolist()

The World Bank does not publish these GDP estimates for every country: many are small territories, but some are countries with limited or disputed data. Any analysis that needs GDP silently leaves them out, and they are not a random group. Before moving on, decide what to do and write it down; at the least, report which countries are missing.

## Worked example: life expectancy by region, weighted by population

A plain average treats every country equally, whether it has 100,000 people or a billion. To describe how long **people** in a region live, weight each country by its population.

In [ ]:
data = (life.merge(pop, on=["country", "code", "year"], how="inner", validate="one_to_one")
            .merge(regions[["code", "region"]], on="code", how="inner", validate="many_to_one"))
data_2020 = data[data["year"] == 2020].copy()

data_2020["weighted"] = data_2020["life_expectancy"] * data_2020["population"]
by_region = data_2020.groupby("region").agg(
    countries=("country", "count"),
    simple_mean=("life_expectancy", "mean"),
    weighted_sum=("weighted", "sum"),
    population=("population", "sum"),
)
by_region["population_weighted_mean"] = by_region["weighted_sum"] / by_region["population"]
by_region[["countries", "simple_mean", "population_weighted_mean"]].round(1)

**Interpretation.** Where the two averages differ, a few very large countries pull the weighted mean toward their own value. In Oceania, most of the region's people live in Australia, so the population-weighted mean is several years higher than the simple mean across its many small island nations. Neither average is wrong; they answer different questions: "a typical country" versus "a typical person". Always say which one you report.

## Reshaping: wide and long

**Long** format has one row per observation (country, year, value) and suits pandas and plotting. **Wide** format has one column per year and suits reading by eye.

In [ ]:
countries = ["Ghana", "Kenya", "Nepal", "Peru", "Tunisia"]
subset = life[life["country"].isin(countries) & life["year"].isin([2000, 2010, 2020])]

wide = subset.pivot(index="country", columns="year", values="life_expectancy")
wide

In [ ]:
long = wide.reset_index().melt(id_vars="country", var_name="year", value_name="life_expectancy")
print(len(long), "rows, which should equal", len(subset))
long.head()

## Your turn

Open `practice-1.4.ipynb`.